# Evaluation of current models

## Dataset 
- mostly inaturalist derived 
- use crops only for classifier to limit background effects 
- detector: Speciesnet (https://github.com/google/cameratrapai#fine-tuning-speciesnet)
- classifier: fine tuned Yolo26 classifier on speciesnet derived crops 
- full species list:

    **rodents**
    - Rattus norvegicus (brown rat)
    - Rattus rattus (black rat)
    - Mus musculus (house mouse)
    - Apodemus agrarius (wood mouse)
    - Apodemus flavicollis (yellow-necked field mouse)
    - Apodemus sylvaticus (woodland vole)
    - Arvicola amphibius (water vole)
    - Microtus arvalis (common vole)
    - Clethrionomys glareolus (red-backed vole)
    - Microtus agrestis (field vole)
    - Crocidura leucodon  (white-toothed shrew)
    - Sorex araneus  (common shrew)
    - Sorex minutus (pygmy shrew)
    - Sorex coronatus (masked shrew)
    
    **rodents in sri lanka**
    - Suncus murinus (Asian house shrew)
    - Bandicota indica (greater bandicoot rat)
    - Bandicota bengalensis (lesser bandicoot rat)
    - Mus booduga (little Indian field mouse)
    - Vandeleuria oleraceus (Indian long-tailed climbing mouse)

    **predators of snakes and rodents (birds, mammals )**
    - Vulpes vulpes (red fox)
    - Buteo buteo (common buzzard)
    - Meles meles (european badger)
    - Ciconia ciconia (white stork)
    - Ardea cinerea (grey heron)

    **snakes**
    - Vipera berus (common european adder)
    - Coronella austriaca (smooth snake)
    - Natrix natrix (grass snake)
    - Natrix helvetica (barred grass snake)

    **snakes in sri lanka**
    - Ptyas mucosa (Indian python)
    - Bungarus caeruleus (blue krait)
    - Naja naja (common cobra)
    - Daboia russelii (Russell's viper)
    - Coelognathus helena (Hooded snake)

    **predators from sri lanka**
    - Spilornis cheela (crested serpent eagle)
    - Varanus bengalensis (Bengal monitor)
    - Centropus sinensis ( oriental darter)
    - Pavo cristatus (peacock)


In [ ]:
%matplotlib inline

In [ ]:
import glob
from pathlib import Path

import pandas as pd
import seaborn as sns

basepath = Path(
    "/mnt/dataLinux/Development/rodent_experiments/datasets/full_dataset/filtered_kept"
)
data = {}
for sp in basepath.iterdir():
    if sp.is_dir():
        data[sp.name] = len(list(glob.glob(str(sp) + "/*.jpg")))

dataset_size_per_species = pd.DataFrame.from_dict(
    data, orient="index", columns=["count"]
).sort_index()

hist = sns.barplot(
    y=dataset_size_per_species.index,
    x=dataset_size_per_species["count"],
    orient="y",
)
hist.figure.set_size_inches(12, 8)

hist.figure.savefig("/mnt/dataLinux/Development/rodent_experiments/datasets/full_dataset/kept_dataset_size_per_species.png", dpi=300)


## Reduced dataset 
**rodents**
- Rattus norvegicus (brown rat)
- Rattus rattus (black rat)
- Mus musculus (house mouse)

**predators of snakes and rodents (birds, mammals)**
- Vulpes vulpes (red fox)
- Ciconia ciconia (white stork)
- Ardea cinerea (grey heron)

**snakes**
- Vipera berus (common european adder)
- Natrix natrix (grass snake)
- Natrix helvetica (barred grass snake)

**snakes in sri lanka**
- Ptyas mucosa (Indian python)
- Naja naja (common cobra)
- Daboia russelii (Russell's viper)
- Coelognathus helena (Hooded snake)

**predators from sri lanka**
- Varanus bengalensis (Bengal monitor)
- Centropus sinensis ( oriental darter)
- Pavo cristatus (peacock)

**other**
- Canis familiaris
- Felis catus
- Car

In [ ]:
import glob
from pathlib import Path

import pandas as pd
import seaborn as sns

basepath = Path(
    "/mnt/dataLinux/Development/rodent_experiments/datasets/full_dataset/filtered_kept"
)

species = [s.lower() for s in [
    "Rattus norvegicus",
    "Rattus rattus",
    "Mus musculus",
    "Vulpes vulpes",
    "Ciconia ciconia",
    "Ardea cinerea",
    # snakes
    "Vipera berus",
    "Natrix natrix",
    "Natrix helvetica",
    # snakes in sri lanka
    "Ptyas mucosa",
    "Bungarus caeruleus",
    "Naja naja",
    "Daboia russelii",
    "Coelognathus helena",
    # predators from sri lanka
    "Varanus bengalensis",
    "Centropus sinensis",
    "Pavo cristatus",
    "Canis familiaris",
    "Felis catus",
    "car",

]]

data = {}
for sp in basepath.iterdir():
    if sp.is_dir() and sp.name.lower() in species:
        data[sp.name] = len(list(glob.glob(str(sp) + "/*.jpg")))

dataset_size_per_species = pd.DataFrame.from_dict(
    data, orient="index", columns=["count"]
).sort_index()

hist = sns.barplot(
    y=dataset_size_per_species.index,
    x=dataset_size_per_species["count"],
    orient="y",
)
hist.figure.set_size_inches(12, 8)

hist.figure.savefig("/mnt/dataLinux/Development/rodent_experiments/datasets/full_dataset/kept_dataset_size_per_species_reduced.png", dpi=300)


so we have a number of species taht are essentially intractable: 
- Sorex coronatus (masked shrew)
- Mus booduga (little Indian field mouse)
- Crocidura leucodon  (white-toothed shrew)
- Bandicota sp. (bandicoot rats)
- Vandeleuria oleraceus (Indian long-tailed climbing mouse)

we would need more data for 'car' and 'empty' classes. We have no 'human' class. 

## Evaluation on California small mammals camera trap dataset

There are 3 relevant species in this dataset: 
- Rattus norvegicus (brown rat)
- Mus musculus (house mouse)
- Rattus rattus (black rat)

We run different trained models on it to see how they do. We run pure speciesnet inference as a benchmark, with and without geofencing

In [ ]:
import json
from pathlib import Path

import yaml
from tqdm import tqdm

import smartrodent


def run_inference_pipeline(config_path):
    with open(config_path, "r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    InferenceType = getattr(smartrodent, config["inference_pipeline"])

    if InferenceType is None:
        raise ValueError(f"Invalid inference_pipeline: {config['inference_pipeline']}")

    inferencepipeline = InferenceType.from_config(config_path)
    input_path_base = Path(config["path"]).resolve()
    output_path = Path(config["output"]).resolve()
    output_path.mkdir(parents=True, exist_ok=True)
    specieslist = config.get("specieslist", None)
    if specieslist is None:
        raise ValueError("specieslist must be given")

    for species in specieslist:
        input_path = input_path_base / species
        output_path_species = output_path / species
        output_path_species.mkdir(parents=True, exist_ok=True)
        images = tqdm(
            list(input_path.iterdir()) if input_path.is_dir() else [input_path]
        )

        results = {}

        for img in images:
            predictions = inferencepipeline.predict(img)
            result = predictions[1]
            drawn_image = predictions[0]
            results[img.name] = result

            Path(output_path_species / "images").mkdir(parents=True, exist_ok=True)
            if drawn_image is not None:
                drawn_image.save(output_path_species / "images" / img.name)
            else:
                pass
                # print(f"Warning: No drawn image for {img.name}")

        with open(output_path_species / "results.json", "w", encoding="utf-8") as file:
            json.dump(results, file, indent=4)


# run speciesnet
# run_inference_pipeline(
#     "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_speciesnet.yaml"
# )
# run_inference_pipeline(
#     "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_speciesnet_ammonitor.yaml"
# )

# run on the configs
configs = [
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_custom_common_s_ammonitor.yaml",
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_custom_common_xl_ammonitor.yaml",
    "/home/hmack/Development/rodent_experiments/configs/inference_pipeline_custom_species_xl_ammonitor.yaml",
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_custom_common_s.yaml",
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_custom_common_xl.yaml",
    "/home/hmack/Development/rodent_experiments/configs/inference_pipeline_custom_genus_s_ammonitor.yaml",
    "/home/hmack/Development/rodent_experiments/configs/inference_pipeline_custom_genus_xl_ammonitor.yaml",
    "/home/hmack/Development/rodent_experiments/configs/inference_pipeline_custom_species_s_ammonitor.yaml",
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_custom_genus_s.yaml",
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_custom_genus_xl.yaml",
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_custom_species_xl.yaml",
    # "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_speciesnet_ammonitor.yaml",
    # "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_speciesnet.yaml",
]

for config_path in configs:
    run_inference_pipeline(config_path)


## Plot evaluation results
We compute precision, recall and f1 score for each species and plot them. We also compute the confusion matrix and plot it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
)
import yaml
import json

def to_dataframe(config_path):
    with open(config_path, "r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    outpath = Path(config["output"]).resolve()
    specieslist = config["specieslist"]

    dfs = []
    for species in specieslist:
        json_path = outpath / species / "results.json"
        if not json_path.exists():
            print(f"Warning: {json_path} does not exist")
            continue

        with open(json_path, "r", encoding="utf-8") as file:
            results = json.load(file)

        for img_name, detections in results.items():
            for det_idx, pred in detections.items():
                if not pred:
                    continue

                # Detect format: SpeciesNet has "label", custom has "result"
                if "result" in pred:
                    # Custom classifier format
                    row = {
                        "image": img_name,
                        "detection_idx": det_idx,
                        "species": species,
                        "format": "custom",
                        "result": pred.get("result"),
                        "confidence": pred.get("confidence"),
                        "top5": pred.get("top5"),
                        "top5_confidence": pred.get("top5_confidence"),
                        "bbox": pred.get("bbox"),
                    }
                elif "label" in pred:
                    # SpeciesNet format
                    classes = pred.get("classifications", {}).get("classes", [])
                    scores = pred.get("classifications", {}).get("scores", [])
                    row = {
                        "image": img_name,
                        "detection_idx": det_idx,
                        "species": species,
                        "format": "speciesnet",
                        "result": pred.get("label"),
                        "confidence": pred.get("conf"),
                        "top5": classes[:5],
                        "top5_confidence": scores[:5],
                        "bbox": pred.get("bbox"),
                        "category": pred.get("category"),
                        "detector_label": pred.get("detector_label"),
                        "detector_confidence": pred.get("detector_confidence"),
                        "classification_source": pred.get("classification_source"),
                    }
                else:
                    continue
                dfs.append(row)

    return pd.DataFrame(dfs) if dfs else pd.DataFrame()

def top_k_accuracy(y_true, y_pred, k=3):
    return np.mean([y_true[i] in y_pred[i][:k] for i in range(len(y_true))])

# only use the configs that are not ammonitor for evaluation, since ammonitor does not have biological labels.

configs = [
"/home/hmack/Development/rodent_experiments/configs/inference_pipeline_custom_common_s_ammonitor.yaml",
"/home/hmack/Development/rodent_experiments/configs/inference_pipeline_custom_common_xl_ammonitor.yaml",
"/home/hmack/Development/rodent_experiments/configs/inference_pipeline_custom_genus_s_ammonitor.yaml",
"/home/hmack/Development/rodent_experiments/configs/inference_pipeline_custom_genus_xl_ammonitor.yaml",
# "/home/hmack/Development/rodent_experiments/configs/inference_pipeline_custom_species_s_ammonitor.yaml",
"/home/hmack/Development/rodent_experiments/configs/inference_pipeline_custom_species_xl_ammonitor.yaml",
]

ammonitor_data = {}
for config in configs:
    print(config)
    df = to_dataframe(config)
    ammonitor_data[Path(config).stem] = df

dataframes = []
for name, data in zip(["ammonitor_data", ], [ammonitor_data,]):
    results = {}
    for model, df in data.items():
        df["species"] = "mouse" # b/c we know the data only contains mice, and Mouse sp., the og. label, just means 'some kind of mouse'.
        true_labels = df["species"]
        pred_labels = df["result"]

        processed_pred_labels = df["result"]
        processed_true_labels = df["species"]
        processed_top_5_labels = df["top5"]

        # compute top1 acc
        top1_acc = accuracy_score(processed_true_labels, processed_pred_labels)

        # compute top5 acc
        top5_acc = top_k_accuracy(processed_true_labels, processed_top_5_labels, k = 5)


        results[model] = {
            "top1_acc": top1_acc,
            "top5_acc": top5_acc
        }

    evaluations = pd.DataFrame.from_dict(results, orient="index")
    evaluations["model_type"] = name
    evaluations["model_name"] = evaluations.index
    dataframes.append(evaluations)

df = pd.concat(dataframes)
df.to_csv("/home/hmack/Development/rodent_experiments/outputs/ammonitor.csv")
df_ammonitor=df


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score


def to_dataframe(config_path):
    with open(config_path, "r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    outpath = Path(config["output"]).resolve()
    specieslist = config["specieslist"]

    dfs = []
    for species in specieslist:
        json_path = outpath / species / "results.json"
        if not json_path.exists():
            print(f"Warning: {json_path} does not exist")
            continue

        with open(json_path, "r", encoding="utf-8") as file:
            results = json.load(file)

        for img_name, detections in results.items():
            for det_idx, pred in detections.items():
                if not pred:
                    continue

                # Detect format: SpeciesNet has "label", custom has "result"
                if "result" in pred:
                    # Custom classifier format
                    row = {
                        "image": img_name,
                        "detection_idx": det_idx,
                        "species": species,
                        "format": "custom",
                        "result": pred.get("result"),
                        "confidence": pred.get("confidence"),
                        "top5": pred.get("top5"),
                        "top5_confidence": pred.get("top5_confidence"),
                        "bbox": pred.get("bbox"),
                    }
                elif "label" in pred:
                    # SpeciesNet format
                    classes = pred.get("classifications", {}).get("classes", [])
                    scores = pred.get("classifications", {}).get("scores", [])
                    row = {
                        "image": img_name,
                        "detection_idx": det_idx,
                        "species": species,
                        "format": "speciesnet",
                        "result": pred.get("label"),
                        "confidence": pred.get("conf"),
                        "top5": classes[:5],
                        "top5_confidence": scores[:5],
                        "bbox": pred.get("bbox"),
                        "category": pred.get("category"),
                        "detector_label": pred.get("detector_label"),
                        "detector_confidence": pred.get("detector_confidence"),
                        "classification_source": pred.get("classification_source"),
                    }
                else:
                    continue
                dfs.append(row)

    return pd.DataFrame(dfs) if dfs else pd.DataFrame()

def top_k_accuracy(y_true, y_pred, k=3):
    return np.mean([y_true[i] in y_pred[i][:k] for i in range(len(y_true))])

# only use the configs that are not ammonitor for evaluation, since ammonitor does not have biological labels.
configs = [
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_speciesnet.yaml",
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_custom_genus_s.yaml",
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_custom_genus_xl.yaml",
    "/mnt/dataLinux/Development/rodent_experiments/configs/inference_pipeline_custom_species_xl.yaml",
]

specieslevel_data = {}
for config in configs:
    if 'species' in config and "ammonitor" not in config:
        df = to_dataframe(config)
        specieslevel_data[Path(config).stem] = df

genuslevel_data = {}
for config in configs:
    if 'genus' in config and "ammonitor" not in config:
        df = to_dataframe(config)
        genuslevel_data[Path(config).stem] = df

print(len(genuslevel_data), len(specieslevel_data))
dataframes = []
for name, data in zip(["specieslevel_data", "genuslevel_data"], [specieslevel_data, genuslevel_data]):
    results = {}
    backstep = 1 if data == specieslevel_data else 2
    label_select = 2 if data == specieslevel_data else 1
    for model, df in data.items():
        true_labels = df["species"]
        pred_labels = df["result"]
        if "category" in df.columns:
            # only get the last element of the name, or perhaps try to find if the label is
            # somewhere to be found in the label if whitespaces are replaced with ;
            processed_pred_labels = [";".join(s.lower().split(";")[-3:-backstep]) for s in df["result"]]
            processed_true_labels = [";".join(s.lower().split(" ")[0:label_select]) for s in df["species"]]
            processed_top_5_labels = [[";".join(s.lower().split(";")[-3:-backstep]).lower() for s in top5] for top5 in df["top5"]]
        else:
            processed_pred_labels = df["result"]
            if data == specieslevel_data:
                processed_true_labels = df["species"]
            else:
                # no lower() here b/c the species names are by convention capitalized and we want to preserve that for the true labels
                processed_true_labels = [";".join(s.split(" ")[0:label_select]) for s in df["species"]]
            processed_top_5_labels = df["top5"]

        # compute top1 acc
        top1_acc = accuracy_score(processed_true_labels, processed_pred_labels)

        # compute top5 acc
        top5_acc = top_k_accuracy(processed_true_labels, processed_top_5_labels, k = 5)


        results[model] = {
            "top1_acc": top1_acc,
            "top5_acc": top5_acc
        }

    evaluations = pd.DataFrame.from_dict(results, orient="index")
    evaluations["model_type"] = name
    evaluations["model_name"] = evaluations.index
    dataframes.append(evaluations)

df = pd.concat(dataframes)
df.to_csv("/home/hmack/Development/rodent_experiments/outputs/species_genuslevel.csv")

df_species = df

df = pd.concat([df_ammonitor, df_species])
df.to_csv("/home/hmack/Development/rodent_experiments/outputs/all.csv")
df

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

df = pd.read_csv("/home/hmack/Development/rodent_experiments/outputs/all.csv")

fig, ax = plt.subplots(2,1, figsize=(12,8), sharex=True)

ax1 = sns.scatterplot(
    df,
    x = "model_name",
    y  = "top1_acc",
    hue = "model_type",
    ax=ax[0],
)

ax2 = sns.scatterplot(
    df,
    x = "model_name",
    y  = "top5_acc",
    hue = "model_type",
    ax=ax[1],
)

labels = [
    "_".join(s.get_text().split("_")[-2:]) for s in ax2.get_xticklabels()

]
ax2.set_xticklabels(labels, rotation=30)

ax1.set_ylim(0.0, 1.0)
ax2.set_ylim(0.0, 1.0)

fig.tight_layout()
fig.savefig("/mnt/dataLinux/Development/rodent_experiments/outputs/model_acc_compared.pdf")
fig


remarks on results: 
- the ammonitor dataset only contains north american mice and does not distinguish biological genera, but only applies a generic 'mouse' label. Hence, this has only been used for the common name model. For the per species models, we will have to aggregate species/genera into what would correspond to a mouse [Apodemus sp., Mus sp.]. 
- the non-ammonitor dataset is the LilA 'California small mammals' dataset, which has a relatvely close-up, top-down setup and contains images for Mus musculus, Rattus rattus and Rattus norvegicus. 
- genusnames are expected to do better than species, b/c species distinctions are often only apparent in particularities of coat coloring on images (A. agrarius vs. A. sylvaticus), head anatomy or posture, which are difficult or impossible to distinguish on greyscale camera trap data with relatively low resolution. 
- speciesnet does badly on classification, which is surprising. It appears that it has a bias towards charismatic species? Apparently, it's also not very good at 
- given the sources of uncertainty in the image acquisition process and the dependency of posture, angle, perspective etc on species identifiability, I would care more about top5 acc than about top1. 

In [ ]:
# ammonitor analysis for species and genus goes here
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

root = Path("/home/hmack/Development/rodent_experiments")
result_paths = {
    "genus_s": root / "outputs/genus_classifier/s/ammonitor_camera_traps/Mouse sp./results.json",
    "genus_xl": root / "outputs/genus_classifier/xl/ammonitor_camera_traps/Mouse sp./results.json",
    "species_s": root / "outputs/species_classifier/s/ammonitor_camera_traps/Mouse sp./results.json",
    "species_xl": root / "outputs/species_classifier/xl/ammonitor_camera_traps/Mouse sp./results.json",
}

rows = []
for model_name, result_path in result_paths.items():
    if not result_path.exists():
        print(f"Skipping {model_name}: {result_path} does not exist")
        continue

    with result_path.open("r", encoding="utf-8") as file:
        model_results = json.load(file)

    for image, detections in model_results.items():
        for detection_idx, prediction in detections.items():
            if prediction and prediction.get("result"):
                rows.append({
                    "model_name": model_name,
                    "image": image,
                    "detection_idx": detection_idx,
                    "result": prediction["result"],
                    "top5": prediction.get("top5", []),
                })

predictions = pd.DataFrame(rows)
print(predictions)
if predictions.empty:
    raise FileNotFoundError("No AMMonitor classifier results were found")

accepted_genera = {"Mus", "Apodemus", "Rattus",}
accepted_species = {
    "Mus musculus",
    "Apodemus agrarius",
    "Apodemus flavicollis",
    "Apodemus sylvaticus",
    "Rattus rattus",
    "Rattus norvegicus",
}

metrics = []
for model_name, model_predictions in predictions.groupby("model_name"):
    if model_name.startswith("species"):
        top1_correct = model_predictions["result"].str.startswith("Mus ")
        top5_correct = model_predictions["top5"].apply(
            lambda labels: any(label in accepted_species for label in labels)
        )
        classifier_type = "species"
    else:
        top1_correct = model_predictions["result"].eq("Mus")
        top5_correct = model_predictions["top5"].apply(
            lambda labels: any(label in accepted_genera for label in labels)
        )
        classifier_type = "genus"

    metrics.append({
        "model_name": model_name,
        "classifier_type": classifier_type,
        "aggregated_top1_acc": top1_correct.mean(),
        "aggregated_top5_acc": top5_correct.mean(),
        "n_predictions": len(model_predictions),
    })

ammonitor_species_genus = pd.DataFrame(metrics)
display(ammonitor_species_genus)

output_dir = root / "outputs"
ammonitor_species_genus.to_csv(
    output_dir / "ammonitor_species_genus_aggregated_accuracy.csv",
    index=False,
)

plot_data = ammonitor_species_genus.melt(
    id_vars=["model_name", "classifier_type"],
    value_vars=["aggregated_top1_acc", "aggregated_top5_acc"],
    var_name="metric",
    value_name="accuracy",
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=plot_data,
    x="model_name",
    y="accuracy",
    hue="metric",
    ax=ax,
)
ax.set_ylim(0, 1)
ax.set_ylabel("Aggregated accuracy")
ax.set_xlabel("Model")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(output_dir / "ammonitor_species_genus_aggregated_accuracy.pdf")
fig


In [ ]:
# california small mammals analysis for common-name classifiers
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

root = Path("/home/hmack/Development/rodent_experiments")
result_dirs = {
    "common_s": root / "outputs/commonnames_classifier/s/california_small_animals",
    "common_xl": root / "outputs/commonnames_classifier/xl/california_small_animals",
}

# Map the dataset's scientific-name labels to the common-name model labels.
data_to_model_label = {
    "Mus musculus": "mouse",
    "Rattus norvegicus": "rat",
    "Rattus rattus": "rat",
}

rows = []
for model_name, result_dir in result_dirs.items():
    for data_label, model_label in data_to_model_label.items():
        result_path = result_dir / data_label / "results.json"
        if not result_path.exists():
            print(f"Skipping missing results: {result_path}")
            continue

        with result_path.open("r", encoding="utf-8") as file:
            model_results = json.load(file)

        for image, detections in model_results.items():
            for detection_idx, prediction in detections.items():
                if prediction and prediction.get("result"):
                    rows.append({
                        "model_name": model_name,
                        "data_label": data_label,
                        "expected_label": model_label,
                        "image": image,
                        "detection_idx": detection_idx,
                        "result": prediction["result"],
                        "top5": prediction.get("top5", []),
                    })

predictions = pd.DataFrame(rows)
if predictions.empty:
    raise FileNotFoundError(
        "No California-small-mammals common-name classifier results were found"
    )

predictions["top1_correct"] = (
    predictions["result"].str.lower() == predictions["expected_label"]
)
predictions["top5_correct"] = predictions.apply(
    lambda row: row["expected_label"]
    in [str(label).lower() for label in row["top5"]],
    axis=1,
)

common_name_metrics = (
    predictions.groupby("model_name")[["top1_correct", "top5_correct"]]
    .mean()
    .rename(columns={
        "top1_correct": "top1_acc",
        "top5_correct": "top5_acc",
    })
    .reset_index()
)
common_name_metrics["n_predictions"] = predictions.groupby("model_name").size().values
display(common_name_metrics)

output_dir = root / "outputs"
common_name_metrics.to_csv(
    output_dir / "california_common_name_accuracy.csv",
    index=False,
)

plot_data = common_name_metrics.melt(
    id_vars=["model_name", "n_predictions"],
    value_vars=["top1_acc", "top5_acc"],
    var_name="metric",
    value_name="accuracy",
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=plot_data,
    x="model_name",
    y="accuracy",
    hue="metric",
    ax=ax,
)
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_xlabel("Model")
fig.tight_layout()
fig.savefig(output_dir / "california_common_name_accuracy.pdf")
fig


# Evaluation on zoo data 

In [ ]:
# evaluation for /Users/hmack/Development/smartrodent_experiments/datasets/rgb needs to go here.
from pathlib import Path
import numpy as np

rng=np.random.default_rng(42)

path = "/Users/hmack/Development/smartrodent_experiments/datasets/rgb/Rat"
imgs = [p for p in Path(path).iterdir() if p.suffix == ".jpg"]
sampled = set(rng.random.choice(
    imgs,
    1000,
    replace=False
))

for img in imgs:
    if img not in sampled:
        img.unlink()



# Data combination and plotting

In [ ]:
import pandas as pd

ammonitor_species = pd.read_csv("/home/hmack/Development/rodent_experiments/outputs/ammonitor_species_genus_aggregated_accuracy.csv")
california_common = pd.read_csv("/home/hmack/Development/rodent_experiments/outputs/california_common_name_accuracy.csv")
species_genus = pd.read_csv("/home/hmack/Development/rodent_experiments/outputs/species_genuslevel.csv")
common = pd.read_csv("/home/hmack/Development/rodent_experiments/outputs/commonnames.csv")
ammonitor = pd.read_csv("/home/hmack/Development/rodent_experiments/outputs/ammonitor.csv")

def unify_accuracy_columns(dataframe, dataset):
    unified = dataframe.rename(columns={
        "aggregated_top1_acc": "top1_acc",
        "aggregated_top5_acc": "top5_acc",
    }).copy()
    unified["original_model_name"] = unified["model_name"]
    unified["model_name"] = (
        unified["model_name"]
        .str.replace("inference_pipeline_custom_", "", regex=False)
        .str.replace("inference_pipeline_", "", regex=False)
        .str.removesuffix("_ammonitor")
    )
    unified.insert(0, "dataset", dataset)
    return unified.loc[:, [
        "dataset",
        "model_name",
        "original_model_name",
        "top1_acc",
        "top5_acc",
    ]]

# The original AMMonitor table evaluates every model against the generic
# "mouse" label. Only its common-name rows are valid here; genus/species
# accuracy comes from the separately aggregated AMMonitor results.
ammonitor_common = ammonitor[
    ammonitor["model_name"].str.contains("_common_")
]

results = pd.concat(
    [
        unify_accuracy_columns(ammonitor_species, "ammonitor"),
        unify_accuracy_columns(ammonitor_common, "ammonitor"),
        unify_accuracy_columns(common, "ammonitor"),
        unify_accuracy_columns(california_common, "california_small_mammals"),
        unify_accuracy_columns(species_genus, "california_small_mammals"),
    ],
    ignore_index=True,
)

# common.csv and the common-name rows in ammonitor.csv contain the same
# AMMonitor measurements. Keep one row per dataset/model combination.
results = (
    results.drop_duplicates()
    .sort_values(["dataset", "model_name"])
    .reset_index(drop=True)
)
results.to_csv("/mnt/dataLinux/Development/rodent_experiments/outputs/all_results_inference.csv", index=False)
results



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

fig, axs = plt.subplots(2,1, figsize=(12,8))
results = pd.read_csv("/mnt/dataLinux/Development/rodent_experiments/outputs/all_results_inference.csv")

sns.barplot(
    data = results,
    x = "dataset",
    y="top1_acc",
    hue = "model_name",
    ax=axs[0]
)

sns.barplot(
    data = results,
    x = "dataset",
    y="top5_acc",
    hue = "model_name",
    ax=axs[1]
)


fig.savefig("/home/hmack/Development/rodent_experiments/outputs/model_acc_compared.pdf")